## 3. Data Preprocessing

This section implements the shared data preparation agreed in Section 2 (EDA).
All three models (Ridge, Random Forest, Gradient Boosting) start from **the same eligible records,
the same chronological split and the same 81 imputed inputs**.

| Step | Learned from data? | Fitted on |
|---|---|---|
| Single-family, single-card eligibility filter | No (row rules) | n/a |
| Chronological split 2017–2023 / 2024 / 2025 | No | n/a |
| Type standardisation (codes → string labels) | No | n/a |
| Median imputation (numerical) | **Yes** | **2017–2023 only** |
| `"Missing"` category (categorical) | No (constant) | 2017–2023 only |
| Scaling + one-hot (Ridge) / ordinal codes (trees) | **Yes** | **2017–2023 only**, inside each model's pipeline |

### 3.1 Configuration and Shared Feature Lists

The feature lists below are written out explicitly (rather than derived from storage types)
so that every member gets the same 53 numerical and 28 categorical inputs regardless of
pandas or pyarrow version.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

TARGET = "meta_sale_price"
SALE_KEY = ["meta_pin", "meta_sale_date", "meta_sale_price"]
TRAIN_YEARS = (2017, 2023)
VAL_YEAR = 2024
TEST_YEAR = 2025
MISSING_LABEL = "Missing"

# Non-input columns kept alongside each split for error analysis (Phase 4).
# They are NOT model inputs.
ANALYSIS_COLUMNS = [
    "meta_pin", "meta_sale_date", "meta_township_code",
    "meta_nbhd_code", "meta_triad_name",
]

NUMERIC_FEATURES = [
    "char_yrblt", "char_bldg_sf", "char_land_sf", "char_beds", "char_rooms",
    "char_fbath", "char_hbath", "char_frpl", "loc_longitude", "loc_latitude",
    "loc_env_flood_fs_factor", "loc_env_airport_noise_dnl",
    "loc_access_cmap_walk_nta_score", "loc_access_cmap_walk_total_score",
    "prox_num_pin_in_half_mile", "prox_num_bus_stop_in_half_mile",
    "prox_num_school_in_half_mile", "prox_num_school_with_rating_in_half_mile",
    "prox_airport_dnl_total", "prox_nearest_bike_trail_dist_ft",
    "prox_nearest_cemetery_dist_ft", "prox_nearest_cta_route_dist_ft",
    "prox_nearest_cta_stop_dist_ft", "prox_nearest_golf_course_dist_ft",
    "prox_nearest_grocery_store_dist_ft", "prox_nearest_hospital_dist_ft",
    "prox_lake_michigan_dist_ft", "prox_nearest_major_road_dist_ft",
    "prox_nearest_metra_route_dist_ft", "prox_nearest_metra_stop_dist_ft",
    "prox_nearest_park_dist_ft", "prox_nearest_railroad_dist_ft",
    "prox_nearest_road_arterial_daily_traffic", "prox_nearest_road_arterial_dist_ft",
    "prox_nearest_road_arterial_lanes", "prox_nearest_road_arterial_speed_limit",
    "prox_nearest_road_collector_daily_traffic", "prox_nearest_road_collector_dist_ft",
    "prox_nearest_road_collector_lanes", "prox_nearest_road_collector_speed_limit",
    "prox_nearest_road_highway_daily_traffic", "prox_nearest_road_highway_dist_ft",
    "prox_nearest_road_highway_lanes", "prox_nearest_secondary_road_dist_ft",
    "prox_nearest_stadium_dist_ft", "prox_nearest_university_dist_ft",
    "prox_nearest_water_dist_ft", "shp_parcel_centroid_dist_ft_sd",
    "shp_parcel_edge_len_ft_sd", "shp_parcel_interior_angle_sd",
    "shp_parcel_mrr_area_ratio", "shp_parcel_mrr_side_ratio",
    "shp_parcel_num_vertices",
]

CATEGORICAL_FEATURES = [
    "char_type_resd", "char_cnst_qlty", "char_attic_fnsh", "char_gar1_att",
    "char_gar1_area", "char_gar1_size", "char_gar1_cnst", "char_attic_type",
    "char_bsmt", "char_ext_wall", "char_heat", "char_bsmt_fin", "char_roof_cnst",
    "char_class", "char_use", "char_porch", "char_air", "loc_property_city",
    "loc_property_zip", "loc_tax_municipality_name",
    "loc_school_elementary_district_geoid", "loc_school_secondary_district_geoid",
    "loc_env_flood_fema_sfha", "loc_env_flood_fs_risk_direction",
    "loc_env_ohare_noise_contour_no_buffer_bool",
    "prox_nearest_road_arterial_surface_type",
    "prox_nearest_road_collector_surface_type",
    "prox_nearest_road_highway_surface_type",
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
assert len(NUMERIC_FEATURES) == 53 and len(CATEGORICAL_FEATURES) == 28

### 3.2 Eligibility Rules and Chronological Split

We re-apply the Section 2.23 rules to the full single-family dataset: one row per
(property ID, sale date, sale price) group, a known card number, and indicators confirming a single card.
These are row-level rules and learn nothing from the data, so applying them before splitting
does not leak information across periods.

In [ ]:
def select_eligible_sales(df: pd.DataFrame) -> pd.DataFrame:
    """Apply the EDA Section 2.23 eligibility rules to the raw dataset."""
    houses = df.loc[df["meta_modeling_group"] == "SF"].copy()
    houses["meta_sale_date"] = pd.to_datetime(houses["meta_sale_date"])
    rows_per_sale = (
        houses.groupby(SALE_KEY, dropna=False)["meta_pin"].transform("size")
    )
    eligible = (
        (rows_per_sale == 1)
        & houses["meta_card_num"].notna()
        & houses["ind_pin_is_multicard"].eq(False)
        & houses["meta_pin_num_cards"].eq(1)
    )
    return houses.loc[eligible].copy()


def chronological_split(eligible: pd.DataFrame):
    """Return (train, validation, test) DataFrames split by sale year."""
    year = eligible["meta_sale_date"].dt.year
    train = eligible.loc[year.between(*TRAIN_YEARS)].copy()
    val = eligible.loc[year == VAL_YEAR].copy()
    test = eligible.loc[year == TEST_YEAR].copy()
    return train, val, test

In [ ]:
raw = pd.read_parquet("../data/training_data.parquet")
print("Raw dataset shape:", raw.shape)

eligible = select_eligible_sales(raw)
train_df, val_df, test_df = chronological_split(eligible)

split_sizes = pd.DataFrame({
    "split": ["Training (2017–2023)", "Validation (2024)", "Test (2025)"],
    "records": [len(train_df), len(val_df), len(test_df)],
    "first_sale": [d["meta_sale_date"].min().date() for d in (train_df, val_df, test_df)],
    "last_sale": [d["meta_sale_date"].max().date() for d in (train_df, val_df, test_df)],
})
display(split_sizes)

# Must match the counts reported in the EDA (Section 2.23)
assert (len(train_df), len(val_df), len(test_df)) == (276_685, 30_545, 31_535)
print("Split sizes match the EDA.")

The split reproduces the EDA exactly: 276,685 training, 30,545 validation and 31,535 test records.

### 3.3 Type Standardisation

Categorical inputs are stored in several ways (pandas categories, Python booleans, and
the numerically stored flood-risk direction code). We convert all of them to plain string labels
so that every encoder treats them identically. For example, the flood-risk code `-1.0` becomes `"-1"`
and boolean flags become `"True"`/`"False"`. Missing values are kept as missing at this stage.

In [ ]:
def _code_to_label(value):
    """Turn a stored code into a clean string label; keep missing as NaN."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    if isinstance(value, (bool, np.bool_)):
        return str(bool(value))
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return str(int(value))          # e.g. -1.0 -> "-1"
    return str(value)


def standardise_types(split: pd.DataFrame) -> pd.DataFrame:
    """Select the 81 inputs; numeric -> float64, categorical -> str labels."""
    X = split[FEATURES].copy()
    X[NUMERIC_FEATURES] = X[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").astype("float64")
    for col in CATEGORICAL_FEATURES:
        X[col] = X[col].astype(object).map(_code_to_label).astype(object)
    return X

In [ ]:
X_train_raw = standardise_types(train_df)
X_val_raw = standardise_types(val_df)
X_test_raw = standardise_types(test_df)

y_train = train_df[TARGET].astype("float64")
y_val = val_df[TARGET].astype("float64")
y_test = test_df[TARGET].astype("float64")

print("Input matrix shapes:", X_train_raw.shape, X_val_raw.shape, X_test_raw.shape)
print("Missing target values:", y_train.isna().sum(), y_val.isna().sum(), y_test.isna().sum())

display(
    X_train_raw[["loc_env_flood_fs_risk_direction", "loc_env_flood_fema_sfha", "char_air"]]
    .apply(lambda s: s.value_counts(dropna=False).to_dict())
    .to_frame("training value counts")
)

### 3.4 Missing-Value Imputation (Fitted on Training Data Only)

As agreed in Section 2.18, numerical gaps are filled with the **training-set median** and categorical gaps
with a separate `"Missing"` label. The imputer is fitted on the 2017–2023 inputs only;
the validation and test sets are transformed with those same training medians.

In [ ]:
def make_shared_imputer() -> ColumnTransformer:
    """Median for numeric, constant 'Missing' for categorical."""
    imputer = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
            ("cat", SimpleImputer(strategy="constant", fill_value=MISSING_LABEL,
                                  missing_values=np.nan), CATEGORICAL_FEATURES),
        ],
        verbose_feature_names_out=False,
    )
    return imputer.set_output(transform="pandas")


def apply_imputer(imputer, X: pd.DataFrame) -> pd.DataFrame:
    out = imputer.transform(X)
    out[NUMERIC_FEATURES] = out[NUMERIC_FEATURES].astype("float64")
    out[CATEGORICAL_FEATURES] = out[CATEGORICAL_FEATURES].astype(str)
    return out[FEATURES]

In [ ]:
imputer = make_shared_imputer()
imputer.fit(X_train_raw)          # learns medians from 2017–2023 only

X_train = apply_imputer(imputer, X_train_raw)
X_val = apply_imputer(imputer, X_val_raw)
X_test = apply_imputer(imputer, X_test_raw)

# Leakage check: the learned medians equal the training medians
learned_medians = pd.Series(
    imputer.named_transformers_["num"].statistics_, index=NUMERIC_FEATURES
)
assert np.allclose(learned_medians, X_train_raw[NUMERIC_FEATURES].median())
print("Imputer fitted on", imputer.named_transformers_["num"].n_features_in_,
      "numerical features using", len(X_train_raw), "training rows.")

missing_before = pd.DataFrame({
    "train": X_train_raw.isna().sum(),
    "validation": X_val_raw.isna().sum(),
    "test": X_test_raw.isna().sum(),
})
missing_before = missing_before.loc[missing_before.sum(axis=1) > 0]
print("Features with at least one missing value before imputation:", len(missing_before))
display(missing_before.sort_values("train", ascending=False).head(10))

print("Missing values after imputation:",
      int(X_train.isna().sum().sum()), int(X_val.isna().sum().sum()), int(X_test.isna().sum().sum()))
display(learned_medians.loc[missing_before.index.intersection(NUMERIC_FEATURES)]
        .head(8).to_frame("training median used"))

### 3.5 Categories Unseen in Training

Because the split is chronological, some categorical levels may appear in 2024 or 2025 but never in
2017–2023. The encoders in Section 3.6 handle these explicitly (an "infrequent" column for Ridge,
code `-1` for the tree models), so validation and test rows are never dropped.

In [ ]:
unseen = []
for col in CATEGORICAL_FEATURES:
    train_levels = set(X_train[col].unique())
    unseen.append({
        "feature": col,
        "train_levels": len(train_levels),
        "val_rows_unseen": int((~X_val[col].isin(train_levels)).sum()),
        "test_rows_unseen": int((~X_test[col].isin(train_levels)).sum()),
    })
unseen = pd.DataFrame(unseen).set_index("feature")
display(unseen.sort_values("train_levels", ascending=False))
print("Features with any unseen level in validation or test:",
      int(((unseen["val_rows_unseen"] + unseen["test_rows_unseen"]) > 0).sum()))

### 3.6 Model-Specific Encoders

Ridge Regression is sensitive to feature scale and cannot use string categories, so it uses
standardised numerical inputs and one-hot encoded categories (levels with fewer than 20 training
rows are pooled). Random Forest and Gradient Boosting are scale-invariant, so they use the
unscaled numerical inputs and ordinal category codes.

These encoders are **not** applied to the exported files. Each model owner places the matching encoder
as the first step of their own `Pipeline`, so that `pipeline.fit(X_train, y_train)` fits the encoder on
training data only.

In [ ]:
def make_ridge_encoder(min_frequency: int = 20) -> ColumnTransformer:
    """Ridge: standardise numerics, one-hot categoricals.

    Levels seen fewer than `min_frequency` times in training are pooled into an
    'infrequent' column; levels unseen in training (2024/2025) map there too.
    """
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                  min_frequency=min_frequency,
                                  sparse_output=True), CATEGORICAL_FEATURES),
        ]
    )


def make_tree_encoder() -> ColumnTransformer:
    """Random Forest / Gradient Boosting: numerics unscaled, ordinal codes.

    Levels unseen in training are encoded as -1.
    """
    return ColumnTransformer(
        transformers=[
            ("num", "passthrough", NUMERIC_FEATURES),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value",
                                   unknown_value=-1, dtype=np.float64),
             CATEGORICAL_FEATURES),
        ]
    )

In [ ]:
ridge_encoder = make_ridge_encoder().fit(X_train)
tree_encoder = make_tree_encoder().fit(X_train)

print("Ridge design matrix:", ridge_encoder.transform(X_val).shape,
      "(53 scaled numerical + one-hot columns)")
print("Tree design matrix: ", tree_encoder.transform(X_val).shape)

# Scaling check: training columns have mean 0 and standard deviation 1
scaled_train = ridge_encoder.named_transformers_["num"].transform(X_train[NUMERIC_FEATURES])
print("Max |mean| of scaled training columns:", float(np.abs(scaled_train.mean(axis=0)).max().round(6)))

### 3.7 Export the Shared Splits

The imputed splits are saved so that every member trains on identical inputs.
The `meta_*` files keep property ID, sale date, township and neighbourhood codes **for error analysis only**
(Phase 4); they are not model inputs. The target is kept on the original dollar scale;
members who model `log(price)` must convert predictions back before calling the shared evaluation function.

In [ ]:
from pathlib import Path
import json

out_dir = Path("../data/stage2_outputs")
out_dir.mkdir(exist_ok=True)

for name, X, y, source in [
    ("train", X_train, y_train, train_df),
    ("val", X_val, y_val, val_df),
    ("test", X_test, y_test, test_df),
]:
    X.to_parquet(out_dir / f"X_{name}.parquet")
    y.to_frame(TARGET).to_parquet(out_dir / f"y_{name}.parquet")
    source[ANALYSIS_COLUMNS].astype(str).to_parquet(out_dir / f"meta_{name}.parquet")

feature_spec = {
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "target": TARGET,
    "split": {"train": "2017-2023", "validation": "2024", "test": "2025"},
    "rows": {"train": len(X_train), "validation": len(X_val), "test": len(X_test)},
    "numeric_imputation_medians": learned_medians.round(6).to_dict(),
    "categorical_missing_label": MISSING_LABEL,
}
(out_dir / "feature_spec.json").write_text(json.dumps(feature_spec, indent=2))

display(pd.DataFrame(
    [(p.name, round(p.stat().st_size / 1e6, 2)) for p in sorted(out_dir.iterdir())],
    columns=["file", "size_MB"],
))

### 3.8 Handoff to Model Training

Each member loads the shared files and wraps the matching encoder in a pipeline. Example:

```python
import sys; sys.path.append("..")   # repo root, where shared_preprocessing.py lives
from shared_preprocessing import load_splits, make_ridge_encoder, make_tree_encoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

X_train, y_train, X_val, y_val, X_test, y_test = load_splits("../data/stage2_outputs")

ridge = make_pipeline(make_ridge_encoder(), Ridge(alpha=1.0))                    # Yuming
rf    = make_pipeline(make_tree_encoder(), RandomForestRegressor(n_jobs=-1))       # Sakura
gb    = make_pipeline(make_tree_encoder(), HistGradientBoostingRegressor())        # Kayla
```

Tune on `X_val`/`y_val`; touch `X_test`/`y_test` only once the final hyperparameters are fixed.
The reload check below confirms that the exported files match the in-memory splits.

In [ ]:
def load_splits(folder: str = "stage2_outputs"):
    """Return X_train, y_train, X_val, y_val, X_test, y_test."""
    from pathlib import Path
    folder = Path(folder)
    out = []
    for name in ["train", "val", "test"]:
        X = pd.read_parquet(folder / f"X_{name}.parquet")
        y = pd.read_parquet(folder / f"y_{name}.parquet")[TARGET]
        out += [X, y]
    return tuple(out)

In [ ]:
X_tr, y_tr, X_va, y_va, X_te, y_te = load_splits("../data/stage2_outputs")
pd.testing.assert_frame_equal(X_tr, X_train)
pd.testing.assert_series_equal(y_te, y_test, check_names=False)
print("Reloaded shapes:", X_tr.shape, X_va.shape, X_te.shape)
print("Reload check passed.")